# 05 — Audit point-in-time 13F security identity

This notebook decides which asset identifier can be used without look-ahead and builds the first audited manager–security matrix candidate. It keeps the filing-reported CUSIP at security grain, separates instrument channels, and never applies a mapping snapshot that was unavailable at the decision time.

The cash-share channel is a holdings-representation baseline, not yet a verified common-stock universe. Raw identifiers and matrix pairs remain local; only aggregate diagnostics are displayed.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_spine_13f import quarter_coverage_audit
from src.full_history_13f import read_parquet_columns
from src.point_in_time_13f import FILING_EVENT_COLUMNS, build_local_quarter_panel
from src.security_master_13f import (
    SECURITY_MASTER_CONTRACT_VERSION,
    build_cash_share_matrix_candidate,
    cached_mapping_snapshot_audit,
    classify_security_rows,
    full_history_security_identity_audit,
    mapping_cutoff_summary,
    security_conflict_audit,
    security_row_audit,
)

PRODUCT_ROOT = PROJECT_ROOT / "data" / "full_history" / "13f" / "v1"
TABLES_ROOT = PRODUCT_ROOT / "tables"
AUDIT_ROOT = PROJECT_ROOT / "data" / "audits"
PRODUCT_ROOT.is_dir(), SECURITY_MASTER_CONTRACT_VERSION

## 1. Full-history identifier evidence

The first run scans only the security-identity columns from every holdings partition and writes an aggregate, fingerprinted cache below the ignored data directory. Later runs reuse that cache. The scan measures reported CUSIP format, reported issuer-CIK coverage, instrument channels, and cross-history identifier conflicts without displaying any identifier.

In [ ]:
full_identity_audit = full_history_security_identity_audit(
    TABLES_ROOT / "holdings",
    cache_directory=AUDIT_ROOT / "13f_security_master",
)
coverage = full_identity_audit.coverage.copy()
coverage["usable_cusip_rows"] = coverage["rows"].where(
    coverage["cusip_status"].eq("usable_reported"), 0
)
coverage["reported_cik_rows"] = coverage["rows"].where(
    coverage["reported_issuer_cik_status"].eq("reported_with_filing"), 0
)
coverage["cash_share_rows"] = coverage["rows"].where(
    coverage["instrument_class"].eq("cash_share_candidate"), 0
)
full_history_quarter_summary = (
    coverage.groupby("report_quarter", dropna=False)
    .agg(
        holding_rows=("rows", "sum"),
        usable_cusip_rows=("usable_cusip_rows", "sum"),
        reported_cik_rows=("reported_cik_rows", "sum"),
        cash_share_rows=("cash_share_rows", "sum"),
    )
    .reset_index()
)
full_history_quarter_summary["usable_cusip_pct"] = (
    100 * full_history_quarter_summary["usable_cusip_rows"]
    / full_history_quarter_summary["holding_rows"]
)
full_history_quarter_summary["reported_cik_pct"] = (
    100 * full_history_quarter_summary["reported_cik_rows"]
    / full_history_quarter_summary["holding_rows"]
)
display(full_history_quarter_summary)
display(full_identity_audit.conflicts)

Cross-channel CUSIP reuse is expected when a manager reports cash shares and options on the same underlying. The typed identifier prevents those exposures from being combined. A CUSIP linked to multiple reported issuer CIKs blocks automatic issuer aggregation, but it does not invalidate a security-level representation.

## 2. Rebuild the first decision-complete point-in-time quarter

The quarter and cutoff are derived from the local filing history. The state machine from notebook 04 resolves visible originals and amendments before any security filter is applied.

In [ ]:
filings = read_parquet_columns(TABLES_ROOT / "filings", FILING_EVENT_COLUMNS)
coverage_window = quarter_coverage_audit(filings)
eligible_quarters = coverage_window.loc[
    coverage_window["eligible_complete_window"]
].copy()
if eligible_quarters.empty:
    raise RuntimeError("No decision-complete quarter is available")

selected_quarter = eligible_quarters.iloc[0]
REPORT_DATE = selected_quarter["report_date_utc"]
DECISION_AT = selected_quarter["information_cutoff_utc"]

HOLDING_COLUMNS = (
    "holding_id",
    "accession_no",
    "manager_cik",
    "period_of_report",
    "filed_at",
    "availability_timestamp_utc",
    "cusip",
    "title_of_class",
    "put_call",
    "shares_or_principal_type",
    "value_usd",
    "shares_or_principal_amount",
    "reported_issuer_cik",
)
panel = build_local_quarter_panel(
    PRODUCT_ROOT,
    report_date=REPORT_DATE,
    decision_at=DECISION_AT,
    holdings_columns=HOLDING_COLUMNS,
)

## 3. Security taxonomy and collision checks

`security_rows_local` contains the raw local identifiers for private Data Wrangler inspection. The displayed tables are aggregate only. Eligibility requires a complete point-in-time manager state, a standard reported CUSIP, the cash-share channel, and a positive reported value.

In [ ]:
security_rows_local = classify_security_rows(panel.holdings_asof)
row_audit = security_row_audit(security_rows_local)
quarter_conflicts = security_conflict_audit(security_rows_local)
display(row_audit)
display(quarter_conflicts)

## 4. Mapping availability gate

Resolved issuer identities are secondary mappings, not facts reported with the holding. A date-only mapping snapshot is admitted only when its date is strictly before the decision date; a same-day snapshot is conservatively unavailable because its intraday time is unknown.

In [ ]:
mapping_summary_local = cached_mapping_snapshot_audit(
    TABLES_ROOT / "issuer_identity_links",
    cache_directory=AUDIT_ROOT / "13f_data_spine",
)
mapping_at_decision = mapping_cutoff_summary(
    mapping_summary_local,
    decision_at=DECISION_AT,
)
display(mapping_at_decision)

## 5. First manager–security matrix candidate

The local pair table aggregates duplicate holding lines within the canonical state, then jointly reapplies the 75 percent retained-weight cap and the 20-security/20-manager degree rules until both hold. Weights are recomputed after the final universe filter. No dense matrix is created.

In [ ]:
matrix_candidate = build_cash_share_matrix_candidate(security_rows_local)
matrix_pairs_local = matrix_candidate.filtered_positions
display(matrix_candidate.audit)
display(matrix_candidate.iterative_log)
display(matrix_candidate.summary)
display(matrix_candidate.checks)

`matrix_pairs_local` is the sparse, local manager–security pair table for Data Wrangler. It contains manager CIKs and reported CUSIPs inside typed identifiers, so it must remain in memory or under the ignored data directory and must never be committed.

## 6. Blocking correctness checks

In [ ]:
checks = pd.concat(
    [
        pd.DataFrame(
            [
                {
                    "check": "matrix source events are available by decision",
                    "violations": int(
                        security_rows_local.loc[
                            security_rows_local["matrix_eligible"],
                            "source_event_available_at_utc",
                        ].gt(panel.decision_at_utc).sum()
                    ),
                    "maximum_absolute_error": 0.0,
                },
                {
                    "check": "matrix uses only the typed cash-share channel",
                    "violations": int(
                        (~matrix_pairs_local["typed_security_id"].str.startswith(
                            "cash_share_candidate:"
                        )).sum()
                    ),
                    "maximum_absolute_error": 0.0,
                },
            ]
        ),
        matrix_candidate.checks,
    ],
    ignore_index=True,
)
assert checks["violations"].eq(0).all(), checks
display(checks)

## 7. Frozen decision and remaining boundary

For the first holdings-representation model:

1. The asset key is the normalized CUSIP reported with the filing, namespaced by instrument channel.
2. The baseline matrix uses the `cash_share_candidate` channel only. Options, debt-like instruments, preferred shares, units, warrants, and explicit funds remain preserved for separate typed models.
3. Reported issuer CIK is audit metadata and a later robustness grain; it is not required for the security-level key.
4. Resolved issuer mappings are excluded unless their snapshot was knowable before the historical decision time. No present-day mapping rewrites history.
5. CUSIP changes remain distinct securities. Company or ultimate-parent consolidation waits for effective-dated corporate-action data.
6. `cash_share_candidate` must not be described as verified common equity. A point-in-time market security master is still required before return joins and trading tests.

If the blocking checks pass, the next milestone is to materialize the sparse matrix pair table for every decision-complete quarter, record its data and code fingerprints, and fit the first binary-ownership factor baseline.